# Demonstrating Different Decoding Strategies with Hugging Face Transformers
In this notebook, we will explore various decoding strategies for text generation using a small encoder-decoder model from Hugging Face's Transformers library. We'll apply these strategies to two different datasets:

- Translation task where deterministic strategies are expected to perform better.
- Summarization task where stochastic strategies might yield more diverse and informative outputs.

The decoding strategies we'll test include:

1. Greedy Search
2. Beam Search
3. Temperature Sampling
4. Top-k Sampling
5. Top-p (Nucleus) Sampling

We'll define a custom ```generate_text``` function to apply these strategies and evaluate their performance using appropriate metrics for each dataset.

Here are some useful links you might want to check:
- [Auto Classes](https://huggingface.co/docs/transformers/model_doc/auto)
- [transformers\AutoModel](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModel)
- [transformers\AutoTokenizer](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoTokenizer)
- [Google's T5](https://arxiv.org/pdf/1910.10683)
- [T5-small](https://huggingface.co/google-t5/t5-small)
- [Flan-T5-small](https://huggingface.co/google/flan-t5-small)

In [3]:
!uv pip install transformers datasets sacrebleu rouge_score evaluate --quiet

In [4]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset
import random
import evaluate

### Load the Pre-trained Model and Tokenizer

We'll use the ```flan-T5-small``` model, which is a small encoder-decoder model suitable for both story_gen and summarization tasks. This model is based on Google's ```t5-small``` model, but fine-tuned on more than 1000 additional tasks covering also more languages.

In [5]:
model_name = 'google/flan-t5-small'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


/home/principal/imat4/NLP_II/labs-github/NLPII-lab3-transformer/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
def generate_text(input_text, strategy='greedy', max_length=50, **kwargs):
    """Generates text based on the specified decoding strategy.

    Args:
        input_text (str): The input text to be processed by the model.
        strategy (str, optional): The decoding strategy to use. Defaults to 'greedy'.
            Options include:
            - 'greedy': Greedy search decoding.
            - 'beam': Beam search decoding.
            - 'temperature': Temperature sampling.
            - 'top-k': Top-k sampling.
            - 'top-p': Top-p (nucleus) sampling.
            - 'contrastive': Contrastive search decoding.
        max_length (int, optional): The maximum length of the generated text. Defaults to 50.
        **kwargs: Additional keyword arguments specific to the decoding strategy.

    Keyword Args:
        num_beams (int, optional): Number of beams for beam search. Defaults to 5.
            Used when `strategy='beam'`.
        temperature (float, optional): Sampling temperature. Defaults to 1.0.
            Used when `strategy='temperature'`.
        top_k (int, optional): The number of highest probability vocabulary tokens to keep for top-k filtering. Defaults to 50.
            Used when `strategy='top-k'` or when `strategy='contrastive'`.
        top_p (float, optional): Cumulative probability for nucleus sampling. Defaults to 0.95.
            Used when `strategy='top-p'`.
        penalty_alpha (float, optional): Contrastive search penalty factor. Defaults to 0.6.
            Used when `strategy='contrastive'`.

    Returns:
        str: The generated text based on the decoding strategy.

    Raises:
        ValueError: If an unknown decoding strategy is specified.
    """
    input_ids = tokenizer.encode(input_text, return_tensors='pt')
    model.eval()
    with torch.no_grad():
        if strategy == 'greedy':
            output_ids = model.generate(input_ids, max_length=max_length)
        elif strategy == 'beam':
            num_beams = kwargs.get('num_beams', 5)
            output_ids = model.generate(
                input_ids, max_length=max_length, num_beams=num_beams, early_stopping=True
            )
        elif strategy == 'temperature':
            temperature = kwargs.get('temperature', 1.0)
            output_ids = model.generate(
                input_ids, max_length=max_length, do_sample=True, temperature=temperature
            )
        elif strategy == 'top-k':
            top_k = kwargs.get('top_k', 50)
            output_ids = model.generate(
                input_ids, max_length=max_length, do_sample=True, top_k=top_k
            )
        elif strategy == 'top-p':
            top_p = kwargs.get('top_p', 0.95)
            output_ids = model.generate(
                input_ids, max_length=max_length, do_sample=True, top_p=top_p
            )
        elif strategy == 'contrastive':
            penalty_alpha = kwargs.get('penalty_alpha', 0.6)
            top_k = kwargs.get('top_k', 4)
            output_ids = model.generate(
                input_ids, max_length=max_length, penalty_alpha=penalty_alpha, top_k=top_k
            )
        else:
            raise ValueError("Unknown strategy: {}".format(strategy))

    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return output_text


### Use case 1: Neural Machine Translation
The WMT16 English-German dataset is a collection of parallel sentences in English and German used for machine translation tasks. It is part of the Conference on Machine story_gen (WMT) shared tasks, which are benchmarks for evaluating machine story_gen systems. The dataset contains professionally translated sentences and covers a variety of topics, making it ideal for training and evaluating story_gen models.

We'll use a subset (1% of the test split) of the dataset for evaluation purposes.

In [7]:
# dataset_nmt = load_dataset('wmt16', 'de-en', split='test[:1%]')
dataset_nmt = load_dataset('wmt/wmt16', 'de-en', split='test[:1%]')


In [8]:
# Randomly select a sample from the dataset
sample = random.choice(dataset_nmt['translation'])
input_text = sample['de']
target_text = sample['en']

print("Input Text:")
print(input_text)
print("\nTarget Text:")
print(target_text)


Input Text:
Lamb war es wichtig zu betonen, dass sein "süßer Hund" aber noch lebe und wahrscheinlich aufgeregt sei, und er sagte, die Familienkontakte der toten Frau könnten auf dem Handy gefunden werden.

Target Text:
Lamb made a point to say his "sweet dog" was there alive and probably upset, and said the dead woman's family contacts could be found on her phone.


We need to preprocess the dataset to prepare it for input into the T5 model. The T5 model expects input in a specific format, including a task prefix. This is because the T5 model is a multi-purpose model, being able to perform several task, so we need to tell it what it must do.

In [9]:
def preprocess_translation(examples: list[dict]) -> dict:
    """Preprocesses a single example for the translation task.

    Args:
        examples list[dict]: A list of dictionaries containing 'de' and 'en' keys with English and German sentences.

    Returns:
        dict[list]: A dictionary of lists with added 'src_texts' and 'tgt_texts' keys for model input and target.

    The function:
    - Adds the task prefix 'translate German to English: ' to the German sentence.
    - Stores the result in 'src_texts'.
    - Copies the English sentence to 'tgt_texts'.
    """
    # Empty dictionary
    texts = {}
    
    # T5 expects a "translate German to English: " prefix
    texts['src_texts'] = ['translate German to English: ' + ex['de'] for ex in examples]
    texts['tgt_texts'] = [ex['en'] for ex in examples]
    return texts

dataset_nmt_preproc = preprocess_translation(dataset_nmt["translation"])


Let's see how we can generate a translation using the t5 model. From a random dataset, we are going to create translations using the different implemented strategies:

In [10]:
# Obtain source and target texts
random_index = random.randrange(len(dataset_nmt_preproc['src_texts']))
random_src_sentence = dataset_nmt_preproc['src_texts'][random_index]
random_tgt_sentence = dataset_nmt_preproc['tgt_texts'][random_index]

# Obtain the translated sentence
translation_greedy= generate_text(random_src_sentence, strategy='greedy')
translation_beam_search = generate_text(random_src_sentence, strategy='beam', num_beams=5)
translation_temperature = generate_text(random_src_sentence, strategy='temperature', temperature=0.7)
translation_top_k = generate_text(random_src_sentence, strategy='top-k', top_k=50)
translation_top_p = generate_text(random_src_sentence, strategy='top-p', top_p=0.95)
translation_contrastive = generate_text(random_src_sentence, strategy='contrastive', top_k=4, penalty_alpha=0.6)

print(random_src_sentence)
print(f"Original translation: \n{random_tgt_sentence}")
print(f"Greedy Search: \n{translation_greedy}")
print(f"Beam Search: \n{translation_beam_search}")
print(f"Temperature Sampling: \n{translation_temperature}")
print(f"Top-k Sampling: \n{translation_top_k}")
print(f"Top-p Sampling: \n{translation_top_p}")
print(f"Contrastive Search: \n{translation_contrastive}")

translate German to English: Innerhalb des Hauses fanden die Beamten die Leiche von Amy Prentiss und eine handgeschriebene Notiz, die auf einen weißen Block gekritzelt war: "Mir tut es so leid, ich wollte, ich könnte es rückgängig machen, ich liebte Amy und sie ist die einzige Frau, die mich jemals liebte". Dies stand nach Angaben der Behörden in dem Brief, und er war von Lamb unterzeichnet.
Original translation: 
Inside the home, officers found Amy Prentiss' body and a hand-written note scribbled on a white legal pad: "I am so very sorry I wish I could take it back I loved Amy and she is the only woman who ever loved me," read the letter authorities say was signed by Lamb.
Greedy Search: 
In the House, the officers of Amy Prentiss and a handwritten note, which was on a white block, said, "I would like to make it a bit more difficult, I would like to make it more
Beam Search: 
Within the house, the officers of Amy Prentiss and a handwritten note, which was written on a white block, sai

- Do you see something different between the deterministic and the stochastic strategies? Try different random sentences.
>>> Write your answer here

Yes, there is a clear difference. Greedy, beam search and contrastive search give almost the same translation, and it is always the same if we run it again. 

The stochastic ones (temperature, top-k and top-p) give a different sentence each time, with different words. This makes them more varied, but they also make more mistakes: they invented things that are not in the original, like "alleged murder" or "bright block", and top-p was the worst one. Beam search also got stuck repeating letters ("Mirrrrrrr") at the end.

In general, the deterministic strategies stay closer to the original meaning, which is better when there is only one correct translation.

### Evaluation Metric: BLEU Score
#### What is BLEU Score?
The BLEU (Bilingual Evaluation Understudy) score is a metric for evaluating the quality of text that has been machine-translated from one language to another. It compares a candidate translation to one or more reference translations and calculates a score based on the overlap of n-grams (contiguous sequences of words).

BLEU-4: Considers up to 4-gram matches between the candidate and reference translations. It provides a balance between precision (matching words) and fluency (maintaining the structure of the language). Using BLEU-4 allows us to capture not just individual word matches (unigrams) but also phrases of up to four words. This makes the evaluation more sensitive to the quality of the translation in terms of both accuracy and fluency.

We'll use the ```sacrebleu``` implementation for a standardized BLEU score calculation (which is BLEU-4). You can check the details [here](https://aclanthology.org/W14-3346.pdf).

In [11]:
bleu_metric = evaluate.load('sacrebleu')

In [12]:
strategies = ['greedy', 'beam', 'temperature', 'top-k', 'top-p', 'contrastive']
results = {}

hyperparameters = {
    'num_beams': 5,
    'temperature': 1.0,
    'top_k': 50,
    'top_p': 0.95,
    'penalty_alpha': 0.6
}

for strategy in strategies:
    print(f"Evaluating strategy: {strategy}")
    predictions = []
    references = []
    for src_text, tgt_text in zip(dataset_nmt_preproc["src_texts"], dataset_nmt_preproc["tgt_texts"]): 
        pred = generate_text(src_text, strategy=strategy, **hyperparameters)
        predictions.append(pred)
        references.append([tgt_text])  # SacreBLEU expects a list of references

    # Compute BLEU-4 score
    bleu = bleu_metric.compute(predictions=predictions, references=references, smooth_method='exp')
    results[strategy] = bleu['score']
    print(f"BLEU-4 score for {strategy}: {bleu['score']:.2f}")


Evaluating strategy: greedy
BLEU-4 score for greedy: 20.99
Evaluating strategy: beam
BLEU-4 score for beam: 21.14
Evaluating strategy: temperature
BLEU-4 score for temperature: 11.03
Evaluating strategy: top-k
BLEU-4 score for top-k: 10.20
Evaluating strategy: top-p
BLEU-4 score for top-p: 10.28
Evaluating strategy: contrastive
BLEU-4 score for contrastive: 19.34


- Seeing the above translations and the BLEU score of the different strategies, which strategy would you choose for this use case?
>>> Write your answer here

I would choose beam search because it got the best BLEU score (21.14). But greedy is almost the same (20.99) and contrastive is a bit lower (19.34) so the difference between these three is small, especially since we only used a few sentences. 

The stochastic strategies (temperature, top-k and top-p) got about half of the score (10-11) which matches what we saw in the translations: they add randomness and change the meaning.

So for translation, where there is one correct answer, a deterministic strategy could be the best choice. Beam search is the safest option, although greedy would also be fine and it is faster.

### Use case 2: Story generation

The ```WritingPrompts``` dataset is a collection of imaginative prompts and corresponding stories from the Reddit community. It contains over 300,000 stories written in response to various prompts, making it suitable for training and evaluating models on creative text generation tasks.

We'll use a subset of the dataset for evaluation purposes.

In [13]:
dataset_st_gen = load_dataset('llm-aes/writing-prompts', split='train[:1%]')

In [14]:
# Randomly select a sample from the dataset
sample = random.choice(dataset_st_gen['prompt'])

print("Sample Text:")
print(sample)

Sample Text:
 You are a normal person who spent your entire life infiltrating the evil Empire . You even became the Emperor 's right hand . The day before you finally topple the Empire , the hero arrives , kills the Emperor , and saves the day .



Now, we have to tell the t5 model to generate text after the input sentence.

In [15]:
def preprocess_story_generation(examples: list[dict]) -> dict:
    """Preprocesses a single example for the story generation task.

    Args:
        examples (list[dict]): A list of dictionaries containing 'prompt' key.

    Returns:
        dict[list]: A dictionary of list with added 'src_texts' for model input.

    The function:
    - Adds the task prefix 'Write a story based on: ' to the prompt.
    """
    # Empty dictionary
    texts = {}
    
    # T5 expects a task prefix
    texts['src_texts'] = ['Write a story based on: ' + ex['prompt'] for ex in examples]
    return texts

dataset_st_gen_preproc = preprocess_story_generation(dataset_st_gen)


As we have no original story to compare how good the model generates a story, you should compare the different decoding strategies by looking at some random stories: 

In [16]:
# Obtain source and target texts
random_index = random.randrange(len(dataset_st_gen_preproc['src_texts']))
random_src_sentence = dataset_st_gen_preproc['src_texts'][random_index]

# Obtain the translated sentence
story_gen_greedy= generate_text(random_src_sentence, max_length=300, strategy='greedy')
story_gen_beam_search = generate_text(random_src_sentence, max_length=300, strategy='beam', num_beams=5)
story_gen_temperature = generate_text(random_src_sentence, max_length=300, strategy='temperature', temperature=0.7)
story_gen_top_k = generate_text(random_src_sentence, max_length=300, strategy='top-k', top_k=50)
story_gen_top_p = generate_text(random_src_sentence, max_length=300, strategy='top-p', top_p=0.95)
story_gen_contrastive = generate_text(random_src_sentence, max_length=300, strategy='contrastive', top_k=4, penalty_alpha=0.6)

print(random_src_sentence)
print(f"Greedy Search: \n{story_gen_greedy}")
print(f"Beam Search: \n{story_gen_beam_search}")
print(f"Temperature Sampling: \n{story_gen_temperature}")
print(f"Top-k Sampling: \n{story_gen_top_k}")
print(f"Top-p Sampling: \n{story_gen_top_p}")
print(f"Contrastive Search: \n{story_gen_contrastive}")

Write a story based on:  The Sphinx has gotten sick of coming up with really good riddles and has resorted to other ways to test humans ' cleverness .

Greedy Search: 
The Sphinx is a sphinx that has been a sphinx for years and has been a sphinx for years. The Sphinx is a sphinx that has been a sphinx for years and has been a sphinx for years. The Sphinx is a sphinx that has been a sphinx for years and has been a sphinx for years.
Beam Search: 
The Sphinx has gotten sick of coming up with really good riddles and has resorted to other ways to test humans'cleverness. The Sphinx has gotten sick of coming up with really good riddles and has resorted to other ways to test humans'cleverness.
Temperature Sampling: 
The Sphinx is a darkly glowing purple spider, the most esoteric spider on the planet. The spider's sphinx will come into contact with the sphinx. The spider has gotten sick of coming up with really good riddles and has resorted to other ways to test humans'cleverness.
Top-k Samplin

- Seeing the above generated stories of the different strategies, which strategy would you choose for this use case?
>>> Write your answer here

I would choose temperature sampling. The deterministic strategies (greedy, beam search and contrastive) get stuck repeating the same sentence and beam search just copies the prompt. 

Top-k and top-p are more creative, but the story loses the thread and goes to other topics (zebras, cats, pets). 

Temperature sampling gave the most coherent story and a good balance between creativity and staying on topic, and that is why I would choose it.

### Sensitivity Analysis

The next code cell compares decoding hyperparameters. It uses fixed, seeded examples, reports BLEU on a tuning subset and checks the strongest candidates on a separate validation subset. For story generation, there is no reference story, so it reports length, lexical diversity and repeated 4-gram rate as diagnostics and prints generations for human inspection. These diagnostics are not a substitute for judging coherence, relevance and story quality.

Run the cell below and use its tables and printed examples to answer:

- Which configuration performs best for use case 1 on the validation BLEU score? What did you observe as the decoding parameters changed?

I would choose beam search with 4 beams because it achieved the highest validation BLEU score (20.38). Greedy search had the best score on the first subset but dropped on validation, while beam search with 4 beams did better on the separate examples. Sampling generally gave lower BLEU and the scores tended to fall as temperature or top-k increased. This makes sense for translation because staying close to the source meaning matters more than producing varied wording. The validation set was small so I would still treat beam search with 4 beams as the best configuration in this experiment rather than a guaranteed winner.

- Which configuration would you choose for use case 2 after inspecting the generated stories and diagnostics? What trade-offs did you observe?

I would choose temperature sampling with temperature 1.1 because it produced the longest stories on average (about 92 words) and had the highest diversity score (0.73), with no repeated 4-grams in these samples. Compared with greedy and beam search it seems to offer more variety, which suits creative writing. A higher temperature can also make the text less predictable and may lead it off topic, so I would check the actual stories for coherence and relevance before deciding it is the best overall. The metrics are useful for comparing variety but cannot measure story quality on their own.

In [18]:
# Sensitivity analysis: 
import numpy as np
import pandas as pd

SENSITIVITY_SEED = 2026
NMT_N = min(16, len(dataset_nmt_preproc['src_texts']))  
nmt_rng = random.Random(SENSITIVITY_SEED)
nmt_indices = nmt_rng.sample(range(len(dataset_nmt_preproc['src_texts'])), NMT_N)
nmt_sources = [dataset_nmt_preproc['src_texts'][i] for i in nmt_indices]
nmt_references = [dataset_nmt_preproc['tgt_texts'][i] for i in nmt_indices]

# Configurations vary one parameter at a time
nmt_configs = [
    ('greedy', {}),
    ('beam', {'num_beams': 2}), ('beam', {'num_beams': 4}), ('beam', {'num_beams': 6}),
    ('contrastive', {'top_k': 4, 'penalty_alpha': 0.2}),
    ('contrastive', {'top_k': 4, 'penalty_alpha': 0.6}),
    ('contrastive', {'top_k': 4, 'penalty_alpha': 0.9}),
    ('temperature', {'temperature': 0.7}), ('temperature', {'temperature': 1.0}),
    ('temperature', {'temperature': 1.3}),
    ('top-k', {'top_k': 10}), ('top-k', {'top_k': 50}), ('top-k', {'top_k': 100}),
    ('top-p', {'top_p': 0.80}), ('top-p', {'top_p': 0.90}), ('top-p', {'top_p': 0.95}),
]

def evaluate_nmt_config(strategy, params, sources=nmt_sources, references=nmt_references):
    
    random.seed(SENSITIVITY_SEED)
    np.random.seed(SENSITIVITY_SEED)
    torch.manual_seed(SENSITIVITY_SEED)
    preds = [generate_text(src, strategy=strategy, max_length=100, **params) for src in sources]
    score = bleu_metric.compute(predictions=preds, references=[[ref] for ref in references], smooth_method='exp')['score']
    return score, preds

nmt_rows = []
for strategy, params in nmt_configs:
    score, _ = evaluate_nmt_config(strategy, params)
    nmt_rows.append({'strategy': strategy, 'parameters': str(params) or 'default', 'BLEU': score})
nmt_sensitivity = pd.DataFrame(nmt_rows).sort_values('BLEU', ascending=False).reset_index(drop=True)
display(nmt_sensitivity)
print(f'Fixed test subset: {NMT_N} examples; seed={SENSITIVITY_SEED}.')

remaining = [i for i in range(len(dataset_nmt_preproc['src_texts'])) if i not in set(nmt_indices)]
valid_rng = random.Random(SENSITIVITY_SEED + 1)
valid_indices = valid_rng.sample(remaining, min(32, len(remaining))) if remaining else nmt_indices
valid_sources = [dataset_nmt_preproc['src_texts'][i] for i in valid_indices]
valid_references = [dataset_nmt_preproc['tgt_texts'][i] for i in valid_indices]
top_candidates = nmt_sensitivity.head(5)
valid_rows = []
for _, row in top_candidates.iterrows():
    # Retrieve the original parameter dictionary without eval/string parsing
    strategy, params = next((s, p) for s, p in nmt_configs
                            if s == row['strategy'] and (str(p) or 'default') == row['parameters'])
    score, _ = evaluate_nmt_config(strategy, params, valid_sources, valid_references)
    valid_rows.append({'strategy': strategy, 'parameters': row['parameters'], 'validation_BLEU': score})
validation_sensitivity = pd.DataFrame(valid_rows).sort_values('validation_BLEU', ascending=False).reset_index(drop=True)
display(validation_sensitivity)

# Story generation has no reference translation. Compare several fixed prompts and
# independent samples: use repetition/diversity as diagnostics, then inspect text.
story_prompts_all = dataset_st_gen_preproc['src_texts']
story_rng = random.Random(SENSITIVITY_SEED)
story_indices = story_rng.sample(range(len(story_prompts_all)), min(2, len(story_prompts_all)))
story_prompts = [story_prompts_all[i] for i in story_indices]
story_configs = [
    ('greedy', {}), ('beam', {'num_beams': 4}),
    ('temperature', {'temperature': 0.7}), ('temperature', {'temperature': 0.9}), ('temperature', {'temperature': 1.1}),
    ('top-k', {'top_k': 20}), ('top-k', {'top_k': 50}),
    ('top-p', {'top_p': 0.85}), ('top-p', {'top_p': 0.95}),
    ('contrastive', {'top_k': 4, 'penalty_alpha': 0.6}),
]

def story_diagnostics(text):
    words = text.lower().split()
    distinct = len(set(words)) / len(words) if words else 0.0
    repeated_3grams = (sum(words[i:i+3] == words[i-3:i] for i in range(3, len(words)))
                       / max(1, len(words)-3))
    return len(words), distinct, repeated_3grams

story_rows = []
story_examples = {}
for strategy, params in story_configs:
    config_key = f'{strategy} {params}' if params else strategy
    outputs = []
    lengths, distincts, repetition_rates = [], [], []
    # Multiple samples per prompt reveal stochastic variability
    for prompt_i, prompt in enumerate(story_prompts):
        for replicate in range(2):
            seed = SENSITIVITY_SEED + prompt_i * 100 + replicate
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
            text = generate_text(prompt, strategy=strategy, max_length=180, **params)
            outputs.append(text)
            length, distinct, repetition = story_diagnostics(text)
            lengths.append(length); distincts.append(distinct); repetition_rates.append(repetition)
    story_examples[config_key] = outputs
    story_rows.append({
        'strategy': strategy, 'parameters': str(params) or 'default',
        'mean_words': np.mean(lengths), 'distinct_1_ratio': np.mean(distincts),
        'repeated_4gram_rate': np.mean(repetition_rates),
    })
story_sensitivity = pd.DataFrame(story_rows).sort_values(
    ['repeated_4gram_rate', 'distinct_1_ratio'], ascending=[True, False]
).reset_index(drop=True)
display(story_sensitivity)

# Inspect candidates on the same prompts: objective metrics alone cannot judge coherence
print('Prompts and sample generations (first prompt, two seeded runs per configuration):')
print(story_prompts[0])
for key, outputs in story_examples.items():
    print(f'\n--- {key} ---')
    print('Sample 1:', outputs[0])
    print('Sample 2:', outputs[1])


,strategy,parameters,BLEU
0,greedy,{},23.437773
1,beam,{'num_beams': 2},22.636040
2,beam,{'num_beams': 4},22.059955
3,contrastive,"{'top_k': 4, 'penalty_alpha': 0.2}",22.027039
4,contrastive,"{'top_k': 4, 'penalty_alpha': 0.9}",21.777046
5,contrastive,"{'top_k': 4, 'penalty_alpha': 0.6}",21.651215
6,beam,{'num_beams': 6},20.772991
7,temperature,{'temperature': 0.7},16.972829
8,top-k,{'top_k': 10},14.672572
9,top-p,{'top_p': 0.8},13.788406


Fixed test subset: 16 examples; seed=2026.


,strategy,parameters,validation_BLEU
0,beam,{'num_beams': 4},20.375129
1,beam,{'num_beams': 2},19.582881
2,contrastive,"{'top_k': 4, 'penalty_alpha': 0.9}",18.242787
3,greedy,{},16.710844
4,contrastive,"{'top_k': 4, 'penalty_alpha': 0.2}",16.278478


,strategy,parameters,mean_words,distinct_1_ratio,repeated_4gram_rate
0,temperature,{'temperature': 1.1},91.75,0.734645,0.000000
1,top-k,{'top_k': 50},89.00,0.703432,0.000000
2,top-k,{'top_k': 20},71.25,0.694009,0.000000
3,top-p,{'top_p': 0.95},73.50,0.691100,0.000000
4,temperature,{'temperature': 0.7},57.00,0.627319,0.000000
5,beam,{'num_beams': 4},48.00,0.445166,0.000000
6,greedy,{},72.00,0.311266,0.000000
7,contrastive,"{'top_k': 4, 'penalty_alpha': 0.6}",81.00,0.227642,0.000000
8,top-p,{'top_p': 0.85},75.75,0.631318,0.002778
9,temperature,{'temperature': 0.9},56.00,0.730742,0.010638


Prompts and sample generations (first prompt, two seeded runs per configuration):
Write a story based on:  You 're a modern , present-day soldier when suddenly you get sent back time to the Battle of Gettysburg . Fully equipped with 21st century weapons , gears , and tactics , describe your experience at trying to survive one of the bloodiest battles in history .


--- greedy ---
Sample 1: During the battle of Gettysburg, the Allied forces fought in the Battle of Gettysburg. The Allied forces were able to capture the enemy, but the Allied forces were unable to capture the enemy. The Allied forces were able to capture the enemy, but the Allied forces were unable to capture the enemy. The Allied forces were able to capture the enemy, but the Allied forces were unable to capture the enemy. The Allied forces were able to capture the enemy, but the Allied forces were able to capture the enemy. The Allied forces were able to capture the enemy, but the Allied forces were able to capture the e